# The data model and dunder methods

Python’s data model is the layer that lets your own classes behave like built-in types. When you call `len(obj)`, compare two objects, use an object in a set, or iterate over it in a loop, Python is really looking for special methods such as `__len__`, `__eq__`, `__hash__`, and `__iter__`.

The power of dunder methods is that they make your abstractions feel natural. The danger is that incorrect implementations create subtle bugs, especially around hashing, equality, ordering, and context management.

As you move through these notebooks, think in terms of contracts: when Python calls a dunder method, what promise is your class making back to the rest of the language?

## Visual model

```text
your code -> len(x) / x == y / for item in x
             -> __len__ / __eq__ / __iter__
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. `__repr__` and `__str__`

Implement `__repr__` on every class you write. It costs one line and it pays
back in every debugging session.

In [ ]:
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

    def __repr__(self) -> str:
        return f"Point(x={self.x!r}, y={self.y!r})"    # for DEVELOPERS

    def __str__(self) -> str:
        return f"({self.x}, {self.y})"                  # for USERS

| | `__repr__` | `__str__` |
|---|---|---|
| Audience | Developers | End users |
| Goal | Unambiguous | Readable |
| Called by | REPL, `repr()`, containers, debuggers, logging | `print()`, `str()`, f-strings |
| Default | `<Point object at 0x7f...>` | Falls back to `__repr__` |
| Ideal | Valid Python that reconstructs the object | Whatever reads best |

Define `__repr__` always; define `__str__` only when the user-facing form
genuinely differs.

**The rule that matters:** a container's `str()` uses its elements' `repr()`.

```text
>>> print([Point(1, 2)])
[Point(x=1, y=2)]           # __repr__, not __str__
```


So a class with only `__str__` still prints as `<object at 0x...>` inside a
list, which is exactly when you most need to see it. And use `!r` inside your
repr: `f"{self.name!r}"` shows `'Ada'` rather than `Ada`, which distinguishes an
empty string from a missing value.

---

## Concept 6. Context managers

In [ ]:
class Timer:
    def __enter__(self) -> "Timer":
        self.start = time.perf_counter()
        return self                    # what `as x` binds

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.elapsed = time.perf_counter() - self.start
        return False                   # False/None: do NOT suppress exceptions

`__exit__` runs **whether or not** an exception occurred — that is the entire
point. Its three arguments are `None, None, None` on a clean exit.

**Returning `True` from `__exit__` swallows the exception.** Almost always
wrong. Do it only when suppression is the explicit purpose, as in
`contextlib.suppress`.

The concise form, which is what you will actually write (Module 15):

In [ ]:
from contextlib import contextmanager

@contextmanager
def timer():
    start = time.perf_counter()
    try:
        yield
    finally:                      # finally, not bare -- runs on exception too
        print(f"{time.perf_counter() - start:.3f}s")

---

## Concept 7. Operators

In [ ]:
class Vector:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

    def __add__(self, other: "Vector") -> "Vector":
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar: float) -> "Vector":
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector(self.x * scalar, self.y * scalar)

    __rmul__ = __mul__            # makes 3 * v work as well as v * 3

    def __neg__(self) -> "Vector":
        return Vector(-self.x, -self.y)

    def __abs__(self) -> float:
        return (self.x**2 + self.y**2) ** 0.5

**How Python resolves `a + b`:**

1. Try `type(a).__add__(a, b)`. If it returns `NotImplemented`, continue.
2. Try `type(b).__radd__(b, a)`. If that also returns `NotImplemented`:
3. `TypeError: unsupported operand type(s)`.

(With one refinement: if `type(b)` is a *subclass* of `type(a)`, the reflected
method is tried first, so a subclass can override its parent's behaviour.)

This is why `NotImplemented` matters. Returning it is how you say "not my
problem" and let the other operand try. Note the trap: `NotImplemented` is
**truthy**, so accidentally returning it from `__eq__` and using the result in an
`if` gives you a silent wrong answer plus a `DeprecationWarning`.

**In-place operators** (`__iadd__` etc.) should mutate and `return self` — for a
mutable type. For an immutable one, omit them and Python falls back to
`__add__` plus rebinding. This is exactly Module 02's list-versus-tuple `+=`
distinction, now from the implementer's side.

Only overload operators where the meaning is obvious. `Vector + Vector` is
clear. `User + User` is not, and a `merge()` method would be better.

---

## Concept 9. The whole map

| Group | Methods |
|---|---|
| Representation | `__repr__` `__str__` `__format__` `__bytes__` |
| Comparison | `__eq__` `__ne__` `__lt__` `__le__` `__gt__` `__ge__` `__hash__` |
| Container | `__len__` `__getitem__` `__setitem__` `__delitem__` `__contains__` `__reversed__` |
| Iteration | `__iter__` `__next__` `__aiter__` `__anext__` |
| Numeric | `__add__` `__sub__` `__mul__` `__truediv__` `__floordiv__` `__mod__` `__pow__` `__neg__` `__abs__` `__round__` and the `__r*__` / `__i*__` variants |
| Conversion | `__bool__` `__int__` `__float__` `__index__` `__complex__` |
| Context | `__enter__` `__exit__` `__aenter__` `__aexit__` |
| Callable | `__call__` |
| Attributes | `__getattr__` `__getattribute__` `__setattr__` `__delattr__` `__dir__` |
| Descriptors | `__get__` `__set__` `__delete__` `__set_name__` |
| Class machinery | `__init__` `__new__` `__init_subclass__` `__class_getitem__` `__slots__` |
| Copying | `__copy__` `__deepcopy__` `__reduce__` |
| Pattern matching | `__match_args__` |

You do not need to memorise this. You need to know it exists, so that when you
want your type to work with some piece of syntax, you look up which method
provides it.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `__repr__` and `__str__`
- Section 2: `__eq__` and `__hash__` are a pair
- Section 3: Ordering
- Section 4: The container protocols
- Section 5: Iteration
- Section 6: Context managers
- Section 7: Operators
- Section 8: `__call__`, `__bool__`, `__format__`
- Section 9: The whole map

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

---

## `OnlyGetItem`

No __iter__, no __contains__, no __len__.

In [ ]:
class OnlyGetItem:
    """No __iter__, no __contains__, no __len__."""

    def __init__(self, data: list[int]) -> None:
        self._data = data

    def __getitem__(self, i: int) -> int:
        print(f"    __getitem__({i!r})")
        return self._data[i]

---

## `OnlyStr`

_OnlyStr_

In [ ]:
class OnlyStr:
    def __str__(self) -> str:
        return "I am a nice string"

---

## `OnlyLen`

_OnlyLen_

In [ ]:
class OnlyLen:
    def __init__(self, n: int) -> None:
        self.n = n

    def __len__(self) -> int:
        print("    __len__ called")
        return self.n

---

## `SelfIterator`

_SelfIterator_

In [ ]:
class SelfIterator:
    def __init__(self, items: list[int]) -> None:
        self._items = items
        self._pos = 0

    def __iter__(self) -> SelfIterator:
        return self                       # the trap

    def __next__(self) -> int:
        if self._pos >= len(self._items):
            raise StopIteration
        self._pos += 1
        return self._items[self._pos - 1]

---

## `Vec`

_Vec_

In [ ]:
class Vec:
    def __init__(self, x: int) -> None:
        self.x = x

    def __mul__(self, other: object) -> object:
        print(f"    Vec.__mul__({other!r})")
        if isinstance(other, int):
            return Vec(self.x * other)
        return NotImplemented

    def __repr__(self) -> str:
        return f"Vec({self.x})"

---

## `Sloppy`

_Sloppy_

In [ ]:
class Sloppy:
    def __eq__(self, other: object) -> bool:
        return False                      # instead of NotImplemented

---

## `Suppressor`

_Suppressor_

In [ ]:
class Suppressor:
    def __enter__(self) -> Suppressor:
        return self

    def __exit__(self, *exc: object) -> bool:
        print(f"    __exit__ got {exc[0]}")
        return True                       # what does this do?

---

## `q01`

_q01_

In [ ]:
def q01() -> None:
    # PREDICTION: dunder?  missing?  output?
    print("q01"); og = OnlyGetItem([10, 20, 30])
    print("   ", list(og))

---

## `q02`

_q02_

In [ ]:
def q02() -> None:
    # PREDICTION:
    print("q02"); og = OnlyGetItem([10, 20, 30])
    print("   ", 20 in og)

---

## `q03`

_q03_

In [ ]:
def q03() -> None:
    # PREDICTION:
    print("q03"); og = OnlyGetItem([10, 20, 30])
    try:
        print("   ", len(og))
    except TypeError as exc:
        print("    TypeError:", exc)

---

## `q04`

_q04_

In [ ]:
def q04() -> None:
    # PREDICTION: what does each of the three lines print?
    print("q04"); o = OnlyStr()
    print("   ", str(o))
    print("   ", repr(o)[:20] + "...")
    print("   ", [o])

---

## `q05`

_q05_

In [ ]:
def q05() -> None:
    # PREDICTION: how many times is __len__ called?
    print("q05"); ol = OnlyLen(0)
    if ol:
        print("    truthy")
    else:
        print("    falsy")

---

## `q06`

_q06_

In [ ]:
def q06() -> None:
    # PREDICTION:
    print("q06"); si = SelfIterator([1, 2, 3])
    print("    first  loop:", [x for x in si])
    print("    second loop:", [x for x in si])

---

## `q07`

_q07_

In [ ]:
def q07() -> None:
    # PREDICTION: does this work? which dunder?
    print("q07"); v = Vec(5)
    print("   ", v * 3)

---

## `q08`

_q08_

In [ ]:
def q08() -> None:
    # PREDICTION: does this work? which dunder is tried, and what then?
    print("q08"); v = Vec(5)
    try:
        print("   ", 3 * v)
    except TypeError as exc:
        print("    TypeError:", exc)

---

## `q09`

_q09_

In [ ]:
def q09() -> None:
    # PREDICTION:
    print("q09"); s = Sloppy()
    print("   ", s == s)
    print("   ", s != s)

---

## `q10`

_q10_

In [ ]:
def q10() -> None:
    # PREDICTION: is the exception raised, suppressed, or something else?
    print("q10")
    with Suppressor():
        raise ValueError("boom")
    print("    we got here")

---

## `q11`

_q11_

In [ ]:
def q11() -> None:
    # PREDICTION: special methods are looked up on the TYPE. What happens here?
    print("q11")

    class Plain:
        pass

    p = Plain()
    p.__len__ = lambda: 42      # type: ignore[method-assign]
    try:
        print("   ", len(p))
    except TypeError as exc:
        print("    TypeError:", exc)

---

## `q12`

_q12_

In [ ]:
def q12() -> None:
    # PREDICTION: which of the two objects' __eq__ runs, and in what order?
    print("q12")

    class A:
        def __eq__(self, other: object) -> bool:
            print("    A.__eq__")
            return NotImplemented

    class B(A):
        def __eq__(self, other: object) -> bool:
            print("    B.__eq__")
            return True

    print("   ", A() == B())

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    for fn in [q01, q02, q03, q04, q05, q06, q07, q08, q09, q10, q11, q12]:
        fn()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.